# EDA — creditcard.csv (Bank Credit Card Transactions)

**Task 1 · Adey Innovations Fraud Detection**

This notebook covers:
1. Data loading & cleaning
2. Univariate analysis of anonymised PCA features
3. Amount and Time analysis
4. Class imbalance quantification


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data_loader import load_creditcard, clean_creditcard
from eda_utils   import (plot_class_imbalance, plot_numeric_distributions,
                          plot_correlation_heatmap)

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print('Imports OK')


## 1. Load Data

In [ ]:
# Update this path if creditcard.csv is stored elsewhere
DATA_DIR = '../data'
df_raw = load_creditcard(f'{DATA_DIR}/creditcard.csv')
print(df_raw.shape)
df_raw.head(3)


In [ ]:
print(df_raw.dtypes)
print()
df_raw.describe().T.round(3)


## 2. Data Cleaning

In [ ]:
# Missing values
missing = df_raw.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "  None — dataset is complete.")

# Duplicates
n_dups = df_raw.duplicated().sum()
print(f"\nDuplicate rows: {n_dups}")


In [ ]:
df = clean_creditcard(df_raw)
print(f"Rows after cleaning: {len(df):,}")


## 3. Class Imbalance

In [ ]:
plot_class_imbalance(df['Class'], title='creditcard.csv — Class Distribution')

counts = df['Class'].value_counts().sort_index()
print(f"Legitimate: {counts[0]:,}  |  Fraud: {counts[1]:,}")
print(f"Fraud rate: {df['Class'].mean()*100:.4f}%  →  Extreme imbalance")
print("\nStandard accuracy of a naive 'always predict 0' model:", f"{counts[0]/len(df)*100:.2f}%")
print("This is why accuracy is a misleading metric here.")


## 4. Time & Amount Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Time
for cls, label, color in [(0, 'Legit', '#2196F3'), (1, 'Fraud', '#E53935')]:
    df[df['Class']==cls]['Time'].plot.kde(ax=axes[0], label=label, color=color, linewidth=2)
axes[0].set_title('Transaction Time Distribution')
axes[0].set_xlabel('Seconds since first transaction')
axes[0].legend()

# Amount
for cls, label, color in [(0, 'Legit', '#2196F3'), (1, 'Fraud', '#E53935')]:
    vals = df[df['Class']==cls]['Amount']
    vals[vals < 500].plot.kde(ax=axes[1], label=label, color=color, linewidth=2)
axes[1].set_title('Transaction Amount Distribution (< $500)')
axes[1].set_xlabel('Amount ($)')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Amount statistics by class
print("Amount statistics by Class:")
print(df.groupby('Class')['Amount'].describe().round(2))


## 5. PCA Feature Analysis (V1–V28)

In [ ]:
# Mean value of each V feature by class — reveals which are most discriminative
V_COLS = [f'V{i}' for i in range(1, 29)]

means = df.groupby('Class')[V_COLS].mean().T
means.columns = ['Legitimate_mean', 'Fraud_mean']
means['diff'] = (means['Fraud_mean'] - means['Legitimate_mean']).abs()
means_sorted = means.sort_values('diff', ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(V_COLS))
ax.bar([v - 0.2 for v in x], means_sorted['Legitimate_mean'], width=0.4,
       label='Legitimate', color='#2196F3', alpha=0.8)
ax.bar([v + 0.2 for v in x], means_sorted['Fraud_mean'], width=0.4,
       label='Fraud', color='#E53935', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(means_sorted.index, rotation=45)
ax.set_title('Mean PCA Feature Value by Class (sorted by absolute difference)')
ax.set_ylabel('Mean value')
ax.legend()
plt.tight_layout()
plt.show()

print("Top 10 most discriminative PCA features:")
print(means_sorted.head(10).round(4))


## 6. Correlation Heatmap

In [ ]:
# Only show Time, Amount, Class and top 10 V features to keep it readable
top_v = means_sorted.head(10).index.tolist()
plot_cols = ['Time', 'Amount'] + top_v + ['Class']
plot_correlation_heatmap(df[plot_cols], title='Correlation Matrix (Time, Amount, Top V features)')


## 7. Save Cleaned Data

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/creditcard_clean.csv', index=False)
print("Saved: data/processed/creditcard_clean.csv")
print(f"Shape: {df.shape}")


## 8. EDA Summary

| Finding | Implication |
|---|---|
| Fraud rate = 0.17% | Extremely imbalanced — SMOTE essential on training set |
| Fraud amounts tend to be smaller | `Amount` is a useful feature post-scaling |
| V4, V11, V14, V17 show biggest mean differences | These will likely dominate feature importance |
| `Time` shows two usage peaks (day patterns) | Include `Time` scaled as a feature |
